=============================================================================
# tDCBAM (PROPOSED) — COMBINED DATASET TRAINING FOR PROTOTYPE
=============================================================================

Trains the proposed **tDCBAM** model (DenseNet121 + CBAM + Relation Network) on
**combined CEDAR + BHSig260 datasets** using a unified 70:15:15 split.

## Key Advantages of Combined Training:

1. **More Training Data** (~284 users total) → Better metric learning
2. **Cross-Dataset Robustness** → Model learns invariant features across writing styles
3. **Improved Generalization** → Web-app will handle diverse user signatures
4. **Single Unified Model** → Simpler deployment vs. per-dataset models

## 2-Step Training Process:

| Step | Task | Data Used |
|------|------|-----------|
| **Step 1: Pretraining** | Train backbone with triplet loss | Combined Train (70%) |
| **Step 2: Meta-Training** | Train relation network episodically | Combined Train (70%) |
| **Validation** | Model selection | Combined Val (15%) |
| **Evaluation** | Final unbiased metrics | Combined Test (15%) |

=============================================================================
# STEP 1: SETUP, IMPORTS, AND REPRODUCIBILITY
=============================================================================

In [1]:
import os
import sys
import json
import random
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

# Set repo root
current_dir = os.path.abspath(os.getcwd())
REPO_ROOT = os.path.abspath(os.path.join(current_dir, '..'))
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

# Import Custom Modules
from models.feature_extractor import DenseNetFeatureExtractor
from models.meta_learner import MetricGenerator
from models.Triplet_Siamese_Similarity_Network import tDCBAM
from losses.triplet_loss import TripletLoss
from utils.model_evaluation import compute_metrics, _plot_det_curve, _plot_far_frr, _plot_confusion_matrix, _plot_score_distribution, _plot_roc_curve
from dataloader.tDCBAM_trainloader import get_pretraining_transforms
import time

# Deterministic Seeding
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f" > [System] Seed set to: {seed}")

seed_everything(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" > [System] Device: {DEVICE}")
if torch.cuda.is_available():
    print(f" > [System] CUDA Device Name: {torch.cuda.get_device_name()}")

/home/lawrence/workspace/thesis/thesis/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


 > [System] Seed set to: 42
 > [System] Device: cuda
 > [System] CUDA Device Name: NVIDIA GeForce RTX 5080


=============================================================================
# STEP 2: CONFIGURATION & DATA PATHS
=============================================================================

**Key Change:** Loads pre-generated 70:15:15 splits from `data/ratio_splits/`
instead of manually loading raw datasets.


In [2]:
# -- Input Configuration --
IMG_SIZE = 224
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE)

# -- Step 1: Pretraining Hyperparameters (Triplet Loss) --
PRETRAIN_BATCH_SIZE = 32
PRETRAIN_EPOCHS = 30
PRETRAIN_PHASE1_EPOCHS = 7   # Frozen backbone (CBAM-only)
PRETRAIN_LR = 1e-4
PRETRAIN_MARGIN = 1.0

# -- Step 2: Meta-Training Hyperparameters --
META_BATCH_SIZE = 16
META_EPOCHS = 70
META_PHASE1_EPOCHS = 20       # Frozen backbone
META_LR_PHASE1 = 1e-3
META_LR_BACKBONE = 1e-5
META_LR_HEAD = 2e-4

# Meta-learning protocol
K_SHOT = 1
N_QUERY_GENUINE = 1
N_QUERY_FORGERY = 1

# Model dimensions
FEATURE_DIM = 1024
EMBEDDING_DIM = 2048          # Concatenated: 1024 + 1024

# Preprocessing & Augmentation
PREPROCESS = True
AUGMENT = True

# -- Split Configuration --
# Use pre-generated 70:15:15 splits from ratio_splits directory
SPLIT_DIR = os.path.join(REPO_ROOT, 'data', 'ratio_splits')
SPLIT_RATIO = '70_15_15'

# Dataset split files
CEDAR_SPLIT_FILE = os.path.join(SPLIT_DIR, f'cedar_split_{SPLIT_RATIO}.json')
BHSIG_BENGALI_SPLIT_FILE = os.path.join(SPLIT_DIR, f'bhsig_bengali_split_{SPLIT_RATIO}.json')
BHSIG_HINDI_SPLIT_FILE = os.path.join(SPLIT_DIR, f'bhsig_hindi_split_{SPLIT_RATIO}.json')

# -- Checkpoints & Output --
CHECKPOINT_DIR = os.path.join(REPO_ROOT, 'checkpoints', 'combined_prototype')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

EVAL_DIR = os.path.join(REPO_ROOT, 'model_evals', 'combined_prototype')
os.makedirs(EVAL_DIR, exist_ok=True)

print(f"[Config] Step 1 -- Pretraining: {PRETRAIN_EPOCHS} epochs, margin={PRETRAIN_MARGIN}")
print(f"[Config] Step 2 -- Meta-Training: {META_EPOCHS} epochs, K-shot={K_SHOT}")
print(f"[Config] Split: 70:15:15 (from pre-generated splits)")
print(f"[Config] Split Directory: {SPLIT_DIR}")
print(f"[Config] Checkpoint Dir: {CHECKPOINT_DIR}")
print(f"[Config] Eval Dir: {EVAL_DIR}")

[Config] Step 1 -- Pretraining: 30 epochs, margin=1.0
[Config] Step 2 -- Meta-Training: 70 epochs, K-shot=1
[Config] Split: 70:15:15 (from pre-generated splits)
[Config] Split Directory: /home/lawrence/workspace/thesis/thesis/data/ratio_splits
[Config] Checkpoint Dir: /home/lawrence/workspace/thesis/thesis/checkpoints/combined_prototype
[Config] Eval Dir: /home/lawrence/workspace/thesis/thesis/model_evals/combined_prototype


=============================================================================
# STEP 3: LOAD AND ORGANIZE COMBINED DATASET
=============================================================================

Loads pre-generated 70:15:15 splits from `data/ratio_splits/` for:
- CEDAR
- BHSig-Bengali  
- BHSig-Hindi

Then merges them into a unified combined dataset with unique user identifiers
to maintain dataset origin tracking.

**Note:** Split files must be generated by `scripts/prepare_split_ratios.py` first.

In [3]:
def merge_splits_from_files(split_files, dataset_names):
    """
    Load pre-generated splits from multiple dataset JSON files and merge them.
    
    Args:
        split_files: dict with keys 'cedar', 'bhsig_bengali', 'bhsig_hindi'
        dataset_names: dict with corresponding labels
    
    Returns:
        train_dict, val_dict, test_dict (merged across datasets)
    """
    train_dict = {}
    val_dict = {}
    test_dict = {}
    
    for ds_key, split_file in split_files.items():
        if not os.path.exists(split_file):
            print(f"   WARNING: Split file not found: {split_file}")
            continue
        
        with open(split_file, 'r') as f:
            split_data = json.load(f)
        
        # Add dataset label to each user ID to ensure uniqueness
        ds_label = dataset_names.get(ds_key, ds_key)
        for split_name in ['train', 'val', 'test']:
            if split_name not in split_data:
                continue
            
            users_in_split = split_data[split_name]
            print(f"   {split_name.upper():5s}: {ds_label:20s} -> {len(users_in_split):3d} users")
            
            for uid, user_data in users_in_split.items():
                # Create unique ID: dataset_username
                unique_uid = f"{ds_label}_{uid}"
                if split_name == 'train':
                    train_dict[unique_uid] = user_data
                elif split_name == 'val':
                    val_dict[unique_uid] = user_data
                elif split_name == 'test':
                    test_dict[unique_uid] = user_data
    
    return train_dict, val_dict, test_dict


# Load pre-generated splits for each dataset at 70:15:15 ratio
print(" > Loading Pre-Generated Splits (70:15:15)")
print(f"   Split Directory: {SPLIT_DIR}\n")

split_files = {
    'cedar': CEDAR_SPLIT_FILE,
    'bhsig_bengali': BHSIG_BENGALI_SPLIT_FILE,
    'bhsig_hindi': BHSIG_HINDI_SPLIT_FILE
}

dataset_names = {
    'cedar': 'CEDAR',
    'bhsig_bengali': 'BHSig_Bengali',
    'bhsig_hindi': 'BHSig_Hindi'
}

# Verify all split files exist
missing_files = [f for f in split_files.values() if not os.path.exists(f)]
if missing_files:
    print(f"   ERROR: Missing split files:")
    for f in missing_files:
        print(f"      - {f}")
    print(f"\n   Please run: python scripts/prepare_split_ratios.py --data_root ./data --output_dir {SPLIT_DIR}")
    raise FileNotFoundError(f"Split files not found in {SPLIT_DIR}")

# Merge splits from all datasets
train_dict, val_dict, test_dict = merge_splits_from_files(split_files, dataset_names)

total_users = len(train_dict) + len(val_dict) + len(test_dict)
print(f"\n > COMBINED DATASET: {total_users} users total (70:15:15 split)")
print(f"\n > SPLIT STATISTICS:")
print(f"   Train: {len(train_dict)} users ({len(train_dict)/total_users*100:.1f}%)")
print(f"   Val:   {len(val_dict)} users ({len(val_dict)/total_users*100:.1f}%)")
print(f"   Test:  {len(test_dict)} users ({len(test_dict)/total_users*100:.1f}%)")

# Count total images per split
train_gen = sum(len(v.get("genuine", [])) for v in train_dict.values())
train_forg = sum(len(v.get("forged", [])) for v in train_dict.values())
val_gen = sum(len(v.get("genuine", [])) for v in val_dict.values())
val_forg = sum(len(v.get("forged", [])) for v in val_dict.values())
test_gen = sum(len(v.get("genuine", [])) for v in test_dict.values())
test_forg = sum(len(v.get("forged", [])) for v in test_dict.values())

print(f"\n > IMAGE STATISTICS:")
print(f"   Train: {train_gen} genuine + {train_forg} forged = {train_gen + train_forg} total")
print(f"   Val:   {val_gen} genuine + {val_forg} forged = {val_gen + val_forg} total")
print(f"   Test:  {test_gen} genuine + {test_forg} forged = {test_gen + test_forg} total")

 > Loading Pre-Generated Splits (70:15:15)
   Split Directory: /home/lawrence/workspace/thesis/thesis/data/ratio_splits

   TRAIN: CEDAR                ->  38 users
   VAL  : CEDAR                ->   8 users
   TEST : CEDAR                ->   9 users
   TRAIN: BHSig_Bengali        ->  70 users
   VAL  : BHSig_Bengali        ->  15 users
   TEST : BHSig_Bengali        ->  15 users
   TRAIN: BHSig_Hindi          -> 112 users
   VAL  : BHSig_Hindi          ->  24 users
   TEST : BHSig_Hindi          ->  24 users

 > COMBINED DATASET: 315 users total (70:15:15 split)

 > SPLIT STATISTICS:
   Train: 220 users (69.8%)
   Val:   47 users (14.9%)
   Test:  48 users (15.2%)

 > IMAGE STATISTICS:
   Train: 5280 genuine + 6372 forged = 11652 total
   Val:   1128 genuine + 1362 forged = 2490 total
   Test:  1152 genuine + 1386 forged = 2538 total


=============================================================================
# STEP 4: DATASET CLASSES FOR TRAINING
=============================================================================

Two dataset classes:
1. **TripletDataset** — For Step 1 (pretraining with triplet loss)
2. **EpisodicDataset** — For Step 2 (meta-training with episodic protocol)

In [4]:
class CombinedTripletDataset(Dataset):
    """
    Generates triplets from combined CEDAR + BHSig260 users.
    Uses hard negative mining: 70% skilled forgeries, 30% random negatives.
    """
    def __init__(self, user_dict, transform=None):
        self.transform = transform
        self.user_genuine_map = {}
        self.user_forged_map = {}
        self.all_genuine_paths = []

        for uid, data in user_dict.items():
            gen_key = next((k for k in data.keys() if k.lower() in ['genuine', 'gen']), None)
            forg_key = next((k for k in data.keys() if k.lower() in ['forged', 'forgeries', 'forg']), None)
            
            gen_paths = data.get(gen_key, []) if gen_key else []
            forg_paths = data.get(forg_key, []) if forg_key else []
            
            if len(gen_paths) >= 2:
                self.user_genuine_map[uid] = gen_paths
                self.user_forged_map[uid] = forg_paths
                self.all_genuine_paths.extend([(p, uid) for p in gen_paths])
        
        self.users = list(self.user_genuine_map.keys())
        self.triplets = []
        self._generate_triplets()
        print(f"   TripletDataset: {len(self.triplets)} triplets from {len(self.users)} users")

    def _generate_triplets(self):
        """Regenerate triplets with hard negative mining."""
        self.triplets = []
        
        for anchor_path, anchor_uid in self.all_genuine_paths:
            positives = [p for p in self.user_genuine_map[anchor_uid] if p != anchor_path]
            if not positives:
                continue
            positive_path = random.choice(positives)
            
            # Hard mining: 70% skilled forgery, 30% random user
            forgeries = self.user_forged_map.get(anchor_uid, [])
            if random.random() < 0.7 and len(forgeries) > 0:
                negative_path = random.choice(forgeries)
            else:
                other_uid = random.choice([u for u in self.users if u != anchor_uid])
                negative_path = random.choice(self.user_genuine_map[other_uid])
            
            self.triplets.append((anchor_path, positive_path, negative_path))

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor_p, pos_p, neg_p = self.triplets[idx]
        anchor = self._load(anchor_p)
        pos = self._load(pos_p)
        neg = self._load(neg_p)
        return anchor, pos, neg, torch.tensor([1], dtype=torch.float32)

    def _load(self, path):
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img


class CombinedEpisodicDataset(Dataset):
    """
    Generates episodes for metric learning from combined dataset.
    Each episode: K support (genuine) + queries (genuine + forged).
    
    NOTE: This dataset now uses get_pretraining_transforms to ensure consistency
    with Step 1 pretraining. Both steps use the same preprocessing pipeline.
    """
    def __init__(self, user_dict, k_shot=1, n_query_genuine=1, n_query_forgery=1, 
                 augment=False, exhaustive_eval=False):
        self.k_shot = k_shot
        self.n_query_genuine = n_query_genuine
        self.n_query_forgery = n_query_forgery
        self.augment = augment
        self.exhaustive_eval = exhaustive_eval
        self.users = []
        self.data = {}
        
        # Use the same transforms as Step 1 for consistency
        # Support images: apply augmentation during training
        self.augment_transform = get_pretraining_transforms(
            input_shape=INPUT_SHAPE, 
            preprocess=PREPROCESS, 
            augment=True
        )
        
        # Query images: no augmentation (clean evaluation)
        self.base_transform = get_pretraining_transforms(
            input_shape=INPUT_SHAPE, 
            preprocess=PREPROCESS, 
            augment=False
        )
        
        for uid, udata in user_dict.items():
            gen_key = next((k for k in udata.keys() if k.lower() in ['genuine', 'gen']), None)
            forg_key = next((k for k in udata.keys() if k.lower() in ['forged', 'forgeries', 'forg']), None)
            
            gen_paths = udata.get(gen_key, []) if gen_key else []
            forg_paths = udata.get(forg_key, []) if forg_key else []
            
            if len(gen_paths) >= k_shot + n_query_genuine and len(forg_paths) >= n_query_forgery:
                self.users.append(uid)
                self.data[uid] = {'genuine': gen_paths, 'forged': forg_paths}
        
        print(f"   EpisodicDataset: {len(self.users)} users (K={k_shot}, Qg={n_query_genuine}, Qf={n_query_forgery})")

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        uid = self.users[idx]
        genuine_paths = self.data[uid]['genuine']
        forgery_paths = self.data[uid]['forged']
        
        # Sample support (K genuine)
        support_paths = random.sample(genuine_paths, self.k_shot)
        
        # Sample query genuine (from remaining)
        remaining_gen = [p for p in genuine_paths if p not in support_paths]
        if self.exhaustive_eval:
            query_gen_paths = remaining_gen
            query_forg_paths = forgery_paths
        else:
            if len(remaining_gen) < self.n_query_genuine:
                query_gen_paths = random.choices(genuine_paths, k=self.n_query_genuine)
            else:
                query_gen_paths = random.sample(remaining_gen, self.n_query_genuine)
            
            # Sample query forged
            if len(forgery_paths) < self.n_query_forgery:
                query_forg_paths = random.choices(forgery_paths, k=self.n_query_forgery)
            else:
                query_forg_paths = random.sample(forgery_paths, self.n_query_forgery)
        
        # Load images
        support_imgs = self._load_batch(support_paths, augment=self.augment)
        query_imgs_gen = self._load_batch(query_gen_paths, augment=False)
        query_imgs_forg = self._load_batch(query_forg_paths, augment=False)
        
        query_imgs = torch.cat([query_imgs_gen, query_imgs_forg], dim=0)
        labels_gen = torch.ones(len(query_imgs_gen), dtype=torch.float32)
        labels_forg = torch.zeros(len(query_imgs_forg), dtype=torch.float32)
        query_labels = torch.cat([labels_gen, labels_forg], dim=0)
        
        return {
            'support_images': support_imgs,
            'query_images': query_imgs,
            'query_labels': query_labels,
            'user_id': str(uid)
        }

    def _load_batch(self, paths, augment=False):
        images = []
        transform = self.augment_transform if augment else self.base_transform
        for path in paths:
            try:
                img = Image.open(path).convert('RGB')
                images.append(transform(img))
            except Exception:
                pass
        return torch.stack(images) if images else torch.empty(0)


print(" > Dataset classes defined")

 > Dataset classes defined


=============================================================================
# STEP 5: PRETRAINING ENGINE (Triplet Loss)
=============================================================================

Trains tDCBAM feature extractor on combined dataset using triplet loss.

In [5]:
def run_pretraining(train_dict, device, checkpoint_path,
                    epochs=30, phase1_epochs=10, lr=1e-4, margin=2.0, batch_size=32):
    """
    Pretraining: Train tDCBAM backbone with triplet loss on combined train data.
    """
    print(f"\n   {'─'*60}")
    print(f"   STEP 1: PRETRAINING (Triplet Loss, Combined Data)")
    print(f"   Epochs: {epochs} (Phase1: {phase1_epochs} frozen)")
    print(f"   Users: {len(train_dict)}")
    print(f"   {'─'*60}")
    
    seed_everything(42)
    
    transform = get_pretraining_transforms(input_shape=INPUT_SHAPE, preprocess=PREPROCESS, augment=AUGMENT)
    dataset = CombinedTripletDataset(train_dict, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
    
    model = tDCBAM(backbone_name='densenet121', output_dim=FEATURE_DIM, pretrained=True).to(device)
    criterion = TripletLoss(margin=margin)
    
    # Phase 1: Freeze backbone, train CBAM only
    fe = model.feature_extractor
    for module in [fe.initial_layers, fe.block1, fe.trans1, fe.block2, fe.trans2,
                   fe.block3, fe.trans3, fe.block4, fe.norm5]:
        for param in module.parameters():
            param.requires_grad = False
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"   Phase 1: {trainable:,}/{total:,} params trainable")
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    best_loss = float('inf')
    
    for epoch in range(epochs):
        # Phase transition
        if epoch == phase1_epochs:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.Adam(model.parameters(), lr=lr * 0.1)
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"   Phase 2: Unfrozen — {trainable:,} params trainable")
        
        model.train()
        epoch_loss = 0.0
        
        for anchor, pos, neg, _ in tqdm(loader, desc=f"Pretrain E{epoch+1}", leave=False):
            anchor, pos, neg = anchor.to(device), pos.to(device), neg.to(device)
            optimizer.zero_grad()
            a_feat, p_feat, n_feat = model(anchor, pos, neg)
            loss = criterion(a_feat, p_feat, n_feat)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(loader)
        phase = 1 if epoch < phase1_epochs else 2
        print(f"   [P{phase}] Epoch {epoch+1:02d}/{epochs} | Triplet Loss: {avg_loss:.4f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.feature_extractor.state_dict(), checkpoint_path)
        
        dataset._generate_triplets()
    
    print(f"   Pretraining complete. Best loss: {best_loss:.4f}")
    print(f"   Weights saved: {os.path.basename(checkpoint_path)}")
    return checkpoint_path


print(" > Pretraining engine defined")

 > Pretraining engine defined


=============================================================================
# STEP 6: META-TRAINING ENGINE (Episodic Learning)
=============================================================================

Trains MetricGenerator on combined data using episodic protocol.

In [6]:
def freeze_backbone(feature_extractor):
    """Freeze DenseNet backbone, keep CBAM + fc trainable."""
    for module in [feature_extractor.initial_layers,
                   feature_extractor.block1, feature_extractor.trans1,
                   feature_extractor.block2, feature_extractor.trans2,
                   feature_extractor.block3, feature_extractor.trans3,
                   feature_extractor.block4, feature_extractor.norm5]:
        for param in module.parameters():
            param.requires_grad = False


def unfreeze_backbone(feature_extractor):
    """Unfreeze all parameters."""
    for param in feature_extractor.parameters():
        param.requires_grad = True


def meta_train_epoch(fe, mg, loader, optimizer, criterion, device):
    """One epoch of episodic meta-training."""
    fe.train()
    mg.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for batch in tqdm(loader, desc="Meta-Train", leave=False):
        support_imgs = batch['support_images'].squeeze(1).to(device)
        query_imgs = batch['query_images'].to(device)
        labels = batch['query_labels'].to(device)
        
        B, N_Q, C, H, W = query_imgs.shape
        query_flat = query_imgs.view(B * N_Q, C, H, W)
        labels_flat = labels.view(B * N_Q).unsqueeze(1)
        support_flat = support_imgs.unsqueeze(1).expand(-1, N_Q, -1, -1, -1).reshape(B * N_Q, C, H, W)
        
        optimizer.zero_grad()
        s_feats = fe(support_flat)
        q_feats = fe(query_flat)
        combined = torch.cat((s_feats, q_feats), dim=1)
        scores = mg(combined)
        loss = criterion(scores, labels_flat)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        preds = (torch.sigmoid(scores) > 0.5).float()
        correct += (preds == labels_flat).sum().item()
        total += labels_flat.size(0)
    
    return running_loss / len(loader), correct / total


def meta_validate(fe, mg, loader, device):
    """Validate meta-learner."""
    fe.eval()
    mg.eval()
    all_labels, all_scores = [], []
    
    with torch.no_grad():
        for batch in loader:
            support_imgs = batch['support_images'].squeeze(1).to(device)
            query_imgs = batch['query_images'].to(device)
            labels = batch['query_labels'].to(device)
            
            B, N_Q, C, H, W = query_imgs.shape
            query_flat = query_imgs.view(B * N_Q, C, H, W)
            labels_flat = labels.view(B * N_Q).unsqueeze(1)
            support_flat = support_imgs.unsqueeze(1).expand(-1, N_Q, -1, -1, -1).reshape(B * N_Q, C, H, W)
            
            s_feats = fe(support_flat)
            q_feats = fe(query_flat)
            combined = torch.cat((s_feats, q_feats), dim=1)
            scores = mg(combined)
            probs = torch.sigmoid(scores)
            
            all_scores.extend(probs.cpu().numpy().flatten())
            all_labels.extend(labels_flat.cpu().numpy().flatten())
    
    return compute_metrics(all_labels, all_scores)


def evaluate_model(fe, mg, loader, device, output_dir=None, silent=False):
    """Final evaluation with optional reporting."""
    metrics = meta_validate(fe, mg, loader, device)
    
    if output_dir and not silent:
        print(f"\n{'='*10} FINAL TEST RESULTS {'='*10}")
        print(f"EER            : {metrics['eer']:.2%}")
        print(f"AUC            : {metrics['auc']:.4f}")
        print(f"Best Threshold : {metrics.get('threshold', 0):.4f}")
        print(f"Accuracy       : {metrics['accuracy']:.2%}")
        print(f"Precision      : {metrics.get('precision', 0):.2%}")
        print(f"Recall         : {metrics['recall']:.2%}")
        print(f"F1-Score       : {metrics.get('f1', 0):.2%}")
        print("="*40)
        
        _plot_roc_curve(metrics, output_dir)
        _plot_score_distribution(metrics, output_dir)
        _plot_confusion_matrix(metrics, output_dir)
        _plot_det_curve(metrics, output_dir)
        _plot_far_frr(metrics, output_dir)
    
    return metrics


def run_meta_training(train_dict, val_dict, test_dict, pretrained_path, device, checkpoint_path,
                      epochs=70, phase1_epochs=15, lr_phase1=1e-3, 
                      lr_backbone=1e-5, lr_head=1e-4, batch_size=16, output_dir=None):
    """
    Meta-training: Trains MetricGenerator using episodic protocol on combined data.
    Train on train_dict, validate on val_dict, final test on test_dict.
    """
    print(f"\n   {'-'*60}")
    print(f"   STEP 2: META-TRAINING (Episodic Learning, Combined Data)")
    print(f"   Epochs: {epochs} (Phase1: {phase1_epochs} frozen)")
    print(f"   Train users: {len(train_dict)} | Val users: {len(val_dict)} | Test users: {len(test_dict)}")
    print(f"   {'-'*60}")
    
    train_set = CombinedEpisodicDataset(train_dict, k_shot=K_SHOT, 
                                        n_query_genuine=N_QUERY_GENUINE, 
                                        n_query_forgery=N_QUERY_FORGERY, augment=True)
    val_set = CombinedEpisodicDataset(val_dict, k_shot=K_SHOT,
                                      n_query_genuine=N_QUERY_GENUINE,
                                      n_query_forgery=N_QUERY_FORGERY, augment=False)
    test_set = CombinedEpisodicDataset(test_dict, k_shot=K_SHOT,
                                       n_query_genuine=N_QUERY_GENUINE,
                                       n_query_forgery=N_QUERY_FORGERY, augment=False, exhaustive_eval=False)
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Initialize models
    fe = DenseNetFeatureExtractor(backbone_name='densenet121', output_dim=FEATURE_DIM).to(device)
    mg = MetricGenerator(embedding_dim=EMBEDDING_DIM).to(device)
    
    # Load pretrained weights
    if os.path.exists(pretrained_path):
        state = torch.load(pretrained_path, map_location=device, weights_only=False)
        if isinstance(state, dict) and 'state_dict' in state:
            state = state['state_dict']
        clean_state = {}
        for k, v in state.items():
            new_k = k.replace('feature_extractor.', '', 1) if k.startswith('feature_extractor.') else k
            clean_state[new_k] = v
        try:
            fe.load_state_dict(clean_state, strict=True)
            print(f"   Loaded pretrained weights (strict)")
        except RuntimeError:
            fe.load_state_dict(clean_state, strict=False)
            print(f"   Loaded pretrained weights (partial)")
    else:
        print(f"   WARNING: No pretrained weights, using ImageNet init")
    
    criterion = nn.BCEWithLogitsLoss()
    best_eer, best_acc, best_metrics = 1.0, 0.0, {}
    
    # Phase 1: Frozen backbone
    freeze_backbone(fe)
    optimizer = optim.AdamW([
        {'params': filter(lambda p: p.requires_grad, fe.parameters()), 'lr': lr_phase1},
        {'params': mg.parameters(), 'lr': lr_phase1}
    ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    for epoch in range(phase1_epochs):
        train_loss, train_acc = meta_train_epoch(fe, mg, train_loader, optimizer, criterion, device)
        val_metrics = meta_validate(fe, mg, val_loader, device)
        val_eer, val_acc = val_metrics['eer'], val_metrics['accuracy']
        
        print(f"   [P1] Epoch {epoch+1:02d}/{phase1_epochs} | Loss: {train_loss:.4f} | "
              f"Acc: {train_acc:.2%} | Val EER: {val_eer:.2%}")
        scheduler.step(val_eer)
        
        if val_eer < best_eer or (val_eer == best_eer and val_acc > best_acc):
            best_eer, best_acc, best_metrics = val_eer, val_acc, val_metrics
            torch.save({
                'feature_extractor': fe.state_dict(),
                'metric_generator': mg.state_dict(),
                'metrics': {k: float(v) for k, v in val_metrics.items() if isinstance(v, (int, float, np.floating))}
            }, checkpoint_path)
    
    # Phase 2: Unfreeze backbone
    unfreeze_backbone(fe)
    optimizer = optim.AdamW([
        {'params': fe.initial_layers.parameters(), 'lr': lr_backbone},
        {'params': fe.block1.parameters(), 'lr': lr_backbone},
        {'params': fe.trans1.parameters(), 'lr': lr_backbone},
        {'params': fe.block2.parameters(), 'lr': lr_backbone},
        {'params': fe.trans2.parameters(), 'lr': lr_backbone},
        {'params': fe.block3.parameters(), 'lr': lr_backbone},
        {'params': fe.trans3.parameters(), 'lr': lr_backbone},
        {'params': fe.block4.parameters(), 'lr': lr_backbone},
        {'params': fe.norm5.parameters(), 'lr': lr_backbone},
        {'params': fe.cbam1.parameters(), 'lr': lr_head},
        {'params': fe.cbam2.parameters(), 'lr': lr_head},
        {'params': fe.cbam3.parameters(), 'lr': lr_head},
        {'params': fe.cbam4.parameters(), 'lr': lr_head},
        {'params': fe.regularized_dense_block.parameters(), 'lr': lr_head},
        {'params': mg.parameters(), 'lr': lr_head}
    ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    
    phase2_epochs = epochs - phase1_epochs
    for epoch in range(phase2_epochs):
        train_loss, train_acc = meta_train_epoch(fe, mg, train_loader, optimizer, criterion, device)
        val_metrics = meta_validate(fe, mg, val_loader, device)
        val_eer, val_acc = val_metrics['eer'], val_metrics['accuracy']
        
        g_epoch = phase1_epochs + epoch + 1
        print(f"   [P2] Epoch {epoch+1:02d}/{phase2_epochs} (G:{g_epoch}) | Loss: {train_loss:.4f} | "
              f"Acc: {train_acc:.2%} | Val EER: {val_eer:.2%}")
        scheduler.step(val_eer)
        
        if val_eer < best_eer or (val_eer == best_eer and val_acc > best_acc):
            best_eer, best_acc, best_metrics = val_eer, val_acc, val_metrics
            torch.save({
                'feature_extractor': fe.state_dict(),
                'metric_generator': mg.state_dict(),
                'metrics': {k: float(v) for k, v in val_metrics.items() if isinstance(v, (int, float, np.floating))}
            }, checkpoint_path)
    
    # Final test evaluation
    final_metrics = best_metrics
    if output_dir:
        if os.path.exists(checkpoint_path):
            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
            if isinstance(checkpoint, dict) and 'feature_extractor' in checkpoint:
                fe.load_state_dict(checkpoint['feature_extractor'], strict=True)
                mg.load_state_dict(checkpoint['metric_generator'], strict=True)
        final_metrics = evaluate_model(fe, mg, test_loader, device, output_dir=output_dir, silent=False)
    
    print(f"   Meta-training complete. Best Val EER: {best_eer:.2%}")
    return final_metrics


print(" > Meta-training engine defined")

 > Meta-training engine defined


=============================================================================
# STEP 7: EXECUTE COMBINED TRAINING PIPELINE
=============================================================================

Run the full 2-step training on combined CEDAR + BHSig260 data.

In [7]:
print(f"\n{'='*70}")
print(f"{'COMBINED PROTOTYPE TRAINING — CEDAR + BHSig260':^70}")
print(f"{'='*70}\n")
_total_start = time.time()

# Create experiment directory
exp_dir = os.path.join(CHECKPOINT_DIR, 'combined_70_15_15')
os.makedirs(exp_dir, exist_ok=True)

# -- STEP 1: Pretraining --
_pretrain_start = time.time()
seed_everything(42)
pretrain_path = os.path.join(exp_dir, "pretrained_backbone.pth")

run_pretraining(
    train_dict=train_dict,
    device=DEVICE,
    checkpoint_path=pretrain_path,
    epochs=PRETRAIN_EPOCHS,
    phase1_epochs=PRETRAIN_PHASE1_EPOCHS,
    lr=PRETRAIN_LR,
    margin=PRETRAIN_MARGIN,
    batch_size=PRETRAIN_BATCH_SIZE
)
_pretrain_sec = round(time.time() - _pretrain_start, 1)
print(f"   ⏱  Pretrain time: {_pretrain_sec:.1f}s")

# -- STEP 2: Meta-Training --
_meta_start = time.time()
seed_everything(42)
meta_ckpt_path = os.path.join(exp_dir, "best_meta_model.pth")

final_metrics = run_meta_training(
    train_dict=train_dict,
    val_dict=val_dict,
    test_dict=test_dict,
    pretrained_path=pretrain_path,
    device=DEVICE,
    checkpoint_path=meta_ckpt_path,
    epochs=META_EPOCHS,
    phase1_epochs=META_PHASE1_EPOCHS,
    lr_phase1=META_LR_PHASE1,
    lr_backbone=META_LR_BACKBONE,
    lr_head=META_LR_HEAD,
    batch_size=META_BATCH_SIZE,
    output_dir=EVAL_DIR
)
_meta_sec = round(time.time() - _meta_start, 1)
_total_sec = round(time.time() - _total_start, 1)
print(f"   ⏱  Meta-train time: {_meta_sec:.1f}s")
print(f"   ⏱  Total time: {_total_sec:.1f}s")

print(f"\n{'='*70}")
print("COMBINED PROTOTYPE TRAINING COMPLETE")
print(f"{'='*70}\n")

print(f"Model weights saved to:")
print(f"  - Feature Extractor: {os.path.join(exp_dir, 'pretrained_backbone.pth')}")
print(f"  - Full Model (for inference): {meta_ckpt_path}")
print(f"\nEvaluation plots saved to: {EVAL_DIR}")


            COMBINED PROTOTYPE TRAINING — CEDAR + BHSig260            

 > [System] Seed set to: 42

   ────────────────────────────────────────────────────────────
   STEP 1: PRETRAINING (Triplet Loss, Combined Data)
   Epochs: 30 (Phase1: 7 frozen)
   Users: 220
   ────────────────────────────────────────────────────────────
 > [System] Seed set to: 42
   TripletDataset: 5280 triplets from 220 users
   Phase 1: 1,139,080/8,092,936 params trainable


Pretrain E1:   0%|          | 0/165 [00:00<?, ?it/s]

   [P1] Epoch 01/30 | Triplet Loss: 2.0751


Pretrain E2:   0%|          | 0/165 [00:00<?, ?it/s]

   [P1] Epoch 02/30 | Triplet Loss: 1.8997


...

   [P2] Epoch 48/50 (G:68) | Loss: 0.0982 | Acc: 96.82% | Val EER: 8.51%


Meta-Train:   0%|          | 0/14 [00:00<?, ?it/s]

   [P2] Epoch 49/50 (G:69) | Loss: 0.1264 | Acc: 95.91% | Val EER: 10.64%


Meta-Train:   0%|          | 0/14 [00:00<?, ?it/s]

   [P2] Epoch 50/50 (G:70) | Loss: 0.1312 | Acc: 95.68% | Val EER: 17.02%

========== FINAL TEST RESULTS ==========
EER            : 14.58%
AUC            : 0.9002
Best Threshold : 0.1039
Accuracy       : 86.46%
Precision      : 85.71%
Recall         : 87.50%
F1-Score       : 86.60%
 > Saved ROC Plot to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_prototype/roc_curve.png
 > Saved Distribution Plot to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_prototype/score_distribution.png
 > Saved Confusion Matrix to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_prototype/confusion_matrix.png
 > Saved DET Curve to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_prototype/det_curve.png
 > Saved FAR/FRR Plot to: /home/lawrence/workspace/thesis/thesis/model_evals/combined_prototype/far_frr.png
   Meta-training complete. Best Val EER: 6.38%
   ⏱  Meta-train time: 193.8s
   ⏱  Total time: 1375.5s

COMBINED PROTOTYPE TRAINING COMPLETE

Model we

=============================================================================
# STEP 8: FINAL RESULTS SUMMARY
=============================================================================

In [8]:
# Display final metrics
print(f"\n{'='*70}")
print(f"{'FINAL TEST PERFORMANCE (Combined CEDAR + BHSig260)':^70}")
print(f"{'='*70}")
print(f"EER            : {final_metrics['eer']:.4f} ({final_metrics['eer']*100:.2f}%)")
print(f"Accuracy       : {final_metrics['accuracy']:.4f} ({final_metrics['accuracy']*100:.2f}%)")
print(f"AUC            : {final_metrics['auc']:.4f}")
print(f"Precision      : {final_metrics.get('precision', 0):.4f}")
print(f"Recall         : {final_metrics['recall']:.4f}")
print(f"F1-Score       : {final_metrics.get('f1', 0):.4f}")
print(f"Best Threshold : {final_metrics.get('threshold', 0):.4f}")
print(f"Pretrain Time  : {_pretrain_sec:.1f}s")
print(f"Meta-Train Time: {_meta_sec:.1f}s")
print(f"Total Time     : {_total_sec:.1f}s")
print(f"{'='*70}\n")

# Save results
results_summary = {
    'dataset': 'Combined (CEDAR + BHSig260)',
    'split': '70:15:15',
    'train_users': len(train_dict),
    'val_users': len(val_dict),
    'test_users': len(test_dict),
    'total_users': total_users,
    'eer': float(final_metrics['eer']),
    'accuracy': float(final_metrics['accuracy']),
    'auc': float(final_metrics['auc']),
    'precision': float(final_metrics.get('precision', 0)),
    'recall': float(final_metrics['recall']),
    'f1': float(final_metrics.get('f1', 0)),
    'threshold': float(final_metrics.get('threshold', 0)),
    'pretrain_sec': _pretrain_sec,
    'meta_sec': _meta_sec,
    'total_sec': _total_sec
}

results_path = os.path.join(CHECKPOINT_DIR, 'combined_prototype_results.json')
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f"> Results saved to: {results_path}\n")


          FINAL TEST PERFORMANCE (Combined CEDAR + BHSig260)          
EER            : 0.1458 (14.58%)
Accuracy       : 0.8646 (86.46%)
AUC            : 0.9002
Precision      : 0.8571
Recall         : 0.8750
F1-Score       : 0.8660
Best Threshold : 0.1039
Pretrain Time  : 1181.7s
Meta-Train Time: 193.8s
Total Time     : 1375.5s

> Results saved to: /home/lawrence/workspace/thesis/thesis/checkpoints/combined_prototype/combined_prototype_results.json



=============================================================================
# STEP 9: MODEL EXPORT FOR PRODUCTION
=============================================================================

The trained model is ready for deployment in your web-app.

## Inference Architecture:

1. **Load Model State**
   ```python
   checkpoint = torch.load('best_meta_model.pth')
   feature_extractor.load_state_dict(checkpoint['feature_extractor'])
   metric_generator.load_state_dict(checkpoint['metric_generator'])
   ```

2. **Inference Pipeline**
   ```
   [Query Signature] → [Feature Extractor] → [Feature Vector]
           ↓
   [Reference Signature] → [Feature Extractor] → [Feature Vector]
           ↓
   [Concatenate Features] → [Metric Generator] → [Similarity Score (0-1)]
           ↓
   [Threshold Comparison] → [Genuine/Forged Decision]
   ```

3. **Key Points**
   - Use **both FeatureExtractor + MetricGenerator** for inference
   - Threshold: Use value from results (~0.5 typically)
   - Input images should be **224×224 RGB**
   - Preprocessing: ImageNet normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

## What You Have:

- ✅ **best_meta_model.pth** - Complete trained model (extract both modules from this)
- ✅ **pretrained_backbone.pth** - Triplet-pretrained feature extractor
- ✅ **combined_prototype_results.json** - Final metrics
- ✅ **Evaluation plots** - ROC, DET, score distributions, confusion matrices

In [9]:
# Example: How to use the model for inference

def load_combined_model(checkpoint_path, device='cuda'):
    """Load the trained combined model for inference."""
    fe = DenseNetFeatureExtractor(backbone_name='densenet121', output_dim=FEATURE_DIM).to(device)
    mg = MetricGenerator(embedding_dim=EMBEDDING_DIM).to(device)
    
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        fe.load_state_dict(checkpoint['feature_extractor'], strict=True)
        mg.load_state_dict(checkpoint['metric_generator'], strict=True)
        print(f"✓ Model loaded from {checkpoint_path}")
    else:
        print(f"✗ Checkpoint not found: {checkpoint_path}")
    
    fe.eval()
    mg.eval()
    return fe, mg


def verify_signature_pair(fe, mg, ref_img_path, query_img_path, threshold=0.5, device='cuda'):
    """
    Verify if query signature matches reference signature.
    
    Args:
        fe: Feature extractor module
        mg: Metric generator module
        ref_img_path: Path to reference (genuine) signature
        query_img_path: Path to query signature
        threshold: Decision threshold (default 0.5)
        device: GPU or CPU
        
    Returns:
        dict with similarity score and verification result
    """
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                     std=[0.229, 0.224, 0.225])
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        normalize
    ])
    
    # Load images
    ref_img = Image.open(ref_img_path).convert('RGB')
    query_img = Image.open(query_img_path).convert('RGB')
    
    # Transform
    ref_tensor = transform(ref_img).unsqueeze(0).to(device)
    query_tensor = transform(query_img).unsqueeze(0).to(device)
    
    # Extract features
    with torch.no_grad():
        ref_feat = fe(ref_tensor)
        query_feat = fe(query_tensor)
        combined = torch.cat((ref_feat, query_feat), dim=1)
        score_logit = mg(combined)
        similarity = torch.sigmoid(score_logit).item()
    
    is_genuine = similarity > threshold
    
    return {
        'similarity_score': similarity,
        'is_genuine': is_genuine,
        'confidence': abs(similarity - 0.5) * 2,  # 0-1 confidence
        'threshold': threshold
    }


# Example usage (uncomment to test):
# fe, mg = load_combined_model(meta_ckpt_path, device=DEVICE)
# result = verify_signature_pair(fe, mg, 
#                                ref_img='path/to/reference.png',
#                                query_img='path/to/query.png',
#                                threshold=0.5)
# print(f"Similarity: {result['similarity_score']:.4f}")
# print(f"Verdict: {'GENUINE' if result['is_genuine'] else 'FORGED'} (confidence: {result['confidence']:.2%})")

print("\n✓ Inference functions defined and ready for use")
print("\nTo use in production:")
print("  1. Call load_combined_model() to load trained weights")
print("  2. Call verify_signature_pair() for each signature comparison")
print("  3. Adjust threshold based on your security requirements")


✓ Inference functions defined and ready for use

To use in production:
  1. Call load_combined_model() to load trained weights
  2. Call verify_signature_pair() for each signature comparison
  3. Adjust threshold based on your security requirements
